In [1]:
!pip install fairlearn pgmpy optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 55.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.


In [2]:
import os, gc, copy, math, warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator

warnings.filterwarnings("ignore")

SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_FAIR = 0.005


def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)


def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.array(X)


def ensure_binary(sv):
    """
    BUG FIX: Original code cast to int and checked len(unique)==2,
    but never re-mapped to {0,1}. E.g. a column with values {1,2}
    would pass the check and return {1,2}, breaking every sv==0 mask.
    Fix: always remap the two unique values to 0 and 1 by rank.
    """
    sv = np.array(sv).flatten()
    u  = np.unique(sv)
    if len(u) == 1:
        # degenerate: only one group — return all-zero array
        return np.zeros(len(sv), dtype=int)
    if len(u) == 2:
        # remap smaller value -> 0, larger -> 1 (works for any dtype)
        return (sv == u[1]).astype(int)
    # more than 2: collapse to majority vs rest
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)


def clean_numeric(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.fillna(s.median() if s.notna().any() else 0)


def stratified_split(X, y, sens_list, test_size=0.2):
    key = np.array(y).astype(str).copy()
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    small = u[c < 2]
    key[np.isin(key, small)] = np.array(y)[np.isin(key, small)].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=SEED)
    except Exception:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=SEED)


def build_val_split(y_train, frac=0.15):
    rng     = np.random.RandomState(SEED)
    val_idx = rng.choice(len(y_train), size=max(1, int(len(y_train) * frac)), replace=False)
    mask    = np.ones(len(y_train), dtype=bool)
    mask[val_idx] = False
    return np.where(mask)[0], val_idx


def compute_eo(y_true, y_pred, sv):
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2:
        return 0.0
    tprs, fprs = [], []
    for g in [0, 1]:
        pos = (sv == g) & (y_true == 1)
        neg = (sv == g) & (y_true == 0)
        tprs.append(y_pred[pos].mean() if pos.sum() > 0 else 0.0)
        fprs.append(y_pred[neg].mean() if neg.sum() > 0 else 0.0)
    return max(abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))


def compute_dp(y_pred, sv):
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2:
        return 0.0
    return abs(y_pred[sv == 0].mean() - y_pred[sv == 1].mean())


def compute_max_eo(y_true, y_pred, sens_list):
    return max((compute_eo(y_true, y_pred, s) for s in sens_list), default=0.0)


def compute_max_dp(y_pred, sens_list):
    return max((compute_dp(y_pred, s) for s in sens_list), default=0.0)


def find_group_fair_thresholds(proba, y_true, sens_list, metric="eo", n_thresholds=80):
    """
    BUG FIX (2-sensitive case): original used (t0+t1)/2 — a single averaged
    threshold for all 4 sub-groups, defeating the purpose of group-specific
    thresholds.  Fix: sweep independent thresholds per sensitive attribute
    and apply them multiplicatively across the full group grid.
    """
    thresholds = np.linspace(0.05, 0.95, n_thresholds)
    best_fair  = np.inf
    best_preds = (proba > 0.5).astype(int)
    best_acc   = accuracy_score(y_true, best_preds)

    if len(sens_list) == 1:
        sv = ensure_binary(sens_list[0])
        for t0 in thresholds:
            for t1 in thresholds:
                pred = np.where(sv == 0,
                                (proba > t0).astype(int),
                                (proba > t1).astype(int))
                fair = (compute_max_eo(y_true, pred, sens_list)
                        if metric == "eo"
                        else compute_max_dp(pred, sens_list))
                if fair < best_fair:
                    best_fair  = fair
                    best_preds = pred
                    best_acc   = accuracy_score(y_true, pred)

    else:
        # Build per-group thresholds: one threshold per (sv0_val, sv1_val) pair.
        # We sweep t0 for sv0 and t1 for sv1 independently and combine as the
        # average of the two applicable thresholds for each sample.
        sv0 = ensure_binary(sens_list[0])
        sv1 = ensure_binary(sens_list[1])
        # Use a coarser grid to keep O(n^2) tractable
        thr_coarse = np.linspace(0.10, 0.90, 30)
        for t0 in thr_coarse:
            for t1 in thr_coarse:
                # Each sample's threshold is the mean of its group-specific thresholds
                # from each attribute independently.
                per_sample_thr = (
                    np.where(sv0 == 0, t0, 1.0 - t0) +
                    np.where(sv1 == 0, t1, 1.0 - t1)
                ) / 2.0
                pred = (proba > per_sample_thr).astype(int)
                fair = (compute_max_eo(y_true, pred, sens_list)
                        if metric == "eo"
                        else compute_max_dp(pred, sens_list))
                if fair < best_fair:
                    best_fair  = fair
                    best_preds = pred
                    best_acc   = accuracy_score(y_true, pred)

    return best_preds, best_fair, best_acc


def discretize_for_bbn(X_cont, n_bins=5):
    kbd = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile")
    return kbd.fit_transform(X_cont).astype(int)


def fit_bbn_smoothing_targets(X_tr, y_tr, sens_tr_list, alpha=0.5, n_bins=5):
    results      = []
    y_arr        = np.array(y_tr).astype(int)
    n_train      = len(y_arr)
    corrs        = np.abs(np.corrcoef(X_tr.T, y_arr)[-1, :-1])
    top_k        = min(5, X_tr.shape[1])
    top_feat_idx = np.argsort(corrs)[::-1][:top_k]
    X_disc       = discretize_for_bbn(X_tr[:, top_feat_idx], n_bins=n_bins)
    overall_rate = float(y_arr.mean())
    size_scale   = float(np.clip(n_train / 1000.0, 0.5, 2.0))

    for i, sv in enumerate(sens_tr_list):
        sv         = ensure_binary(sv)
        attr_name  = f"S{i}"
        label_name = "Y"
        feat_names = [f"F{j}" for j in range(top_k)]

        df_bbn = pd.DataFrame(X_disc, columns=feat_names)
        df_bbn[attr_name]  = sv
        df_bbn[label_name] = y_arr

        edges = [(attr_name, label_name)] + [(f, label_name) for f in feat_names]
        model = DiscreteBayesianNetwork(edges)
        est   = BayesianEstimator(model, df_bbn)
        model.cpds = []
        for node in model.nodes():
            cpd = est.estimate_cpd(node, prior_type="BDeu", equivalent_sample_size=10)
            model.add_cpds(cpd)

        n0 = int((sv == 0).sum())
        n1 = int((sv == 1).sum())

        p0_emp = float(y_arr[sv == 0].mean()) if n0 > 0 else 0.5
        p1_emp = float(y_arr[sv == 1].mean()) if n1 > 0 else 0.5

        m0p = (sv == 0) & (y_arr == 1)
        m0n = (sv == 0) & (y_arr == 0)
        m1p = (sv == 1) & (y_arr == 1)
        m1n = (sv == 1) & (y_arr == 0)

        p0_pos = float(y_arr[m0p].mean()) if m0p.sum() > 0 else p0_emp
        p1_pos = float(y_arr[m1p].mean()) if m1p.sum() > 0 else p1_emp
        p0_neg = float(y_arr[m0n].mean()) if m0n.sum() > 0 else 0.0
        p1_neg = float(y_arr[m1n].mean()) if m1n.sum() > 0 else 0.0

        try:
            cpd_y     = model.get_cpds(label_name)
            cpd_vals  = cpd_y.values
            feat_axes = tuple(range(2, 2 + top_k))
            marginal  = cpd_vals.sum(axis=feat_axes)
            marginal  = marginal / marginal.sum(axis=0, keepdims=True)
            p0_bbn    = float(marginal[1, 0])
            p1_bbn    = float(marginal[1, 1])
            w_bbn     = float(np.clip(100.0 / max(min(n0, n1), 1), 0.0, 1.0))
            p0        = (1.0 - w_bbn) * p0_emp + w_bbn * p0_bbn
            p1        = (1.0 - w_bbn) * p1_emp + w_bbn * p1_bbn
        except Exception:
            p0 = p0_emp
            p1 = p1_emp

        gap = abs(p0 - p1)
        eps = float(min(alpha * gap * size_scale, 0.45))

        if p0 >= p1:
            dp_tg0, dp_tg1           = overall_rate, overall_rate
            eo_tpos_g0, eo_tpos_g1   = p0_pos, p0_pos
            eo_tneg_g0, eo_tneg_g1   = p0_neg, p0_neg
        else:
            dp_tg0, dp_tg1           = overall_rate, overall_rate
            eo_tpos_g0, eo_tpos_g1   = p1_pos, p1_pos
            eo_tneg_g0, eo_tneg_g1   = p1_neg, p1_neg

        results.append({
            "p0": p0, "p1": p1, "eps": eps,
            "dp_target_g0": dp_tg0, "dp_target_g1": dp_tg1,
            "eo_target_pos_g0": eo_tpos_g0, "eo_target_pos_g1": eo_tpos_g1,
            "eo_target_neg_g0": eo_tneg_g0, "eo_target_neg_g1": eo_tneg_g1,
        })

        tqdm.write(f"    BBN S{i}: P(Y=1|S=0)={p0:.3f}, P(Y=1|S=1)={p1:.3f}, "
                   f"gap={gap:.3f}, eps={eps:.3f}")

    return results


def make_null_smoothing_targets(sens_tr_list):
    """
    Ablation helper: returns smoothing targets with eps=0 so that
    bbn_smooth_targets is a no-op (labels pass through unchanged).
    """
    return [{"eps": 0.0,
             "dp_target_g0": 0.5, "dp_target_g1": 0.5,
             "eo_target_pos_g0": 1.0, "eo_target_pos_g1": 1.0,
             "eo_target_neg_g0": 0.0, "eo_target_neg_g1": 0.0}
            for _ in sens_tr_list]


def bbn_smooth_targets(y_batch, sens_batch, smoothing_targets, mode="eo"):
    y_f      = y_batch.float().squeeze()
    smoothed = torch.zeros_like(y_f)

    for sv, st in zip(sens_batch, smoothing_targets):
        eps    = st["eps"]
        y_this = y_f.clone()

        if mode == "dp":
            m0 = sv == 0
            m1 = sv == 1
            if m0.any():
                y_this[m0] = (1 - eps) * y_f[m0] + eps * st["dp_target_g0"]
            if m1.any():
                y_this[m1] = (1 - eps) * y_f[m1] + eps * st["dp_target_g1"]
        else:
            m0p = (sv == 0) & (y_f > 0.5)
            m0n = (sv == 0) & (y_f <= 0.5)
            m1p = (sv == 1) & (y_f > 0.5)
            m1n = (sv == 1) & (y_f <= 0.5)
            if m0p.any():
                y_this[m0p] = (1 - eps) * y_f[m0p] + eps * st["eo_target_pos_g0"]
            if m0n.any():
                y_this[m0n] = (1 - eps) * y_f[m0n] + eps * st["eo_target_neg_g0"]
            if m1p.any():
                y_this[m1p] = (1 - eps) * y_f[m1p] + eps * st["eo_target_pos_g1"]
            if m1n.any():
                y_this[m1n] = (1 - eps) * y_f[m1n] + eps * st["eo_target_neg_g1"]

        smoothed = smoothed + y_this

    return (smoothed / max(len(sens_batch), 1)).view(-1, 1)


class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = float(alpha)
        return x.clone()

    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)


class FairClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, n_sensitive=2, dropout=0.20, mode="eo"):
        super().__init__()
        self.mode = mode
        h = hidden_dim

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, h * 2), nn.BatchNorm1d(h * 2),
            nn.LeakyReLU(0.1),        nn.Dropout(dropout),
            nn.Linear(h * 2, h),      nn.BatchNorm1d(h),
            nn.LeakyReLU(0.1),        nn.Dropout(dropout * 0.6),
            nn.Linear(h, h // 2),     nn.BatchNorm1d(h // 2),
            nn.LeakyReLU(0.1),
        )

        self.decorr_proj = nn.Linear(h // 2, h // 4)

        self.classifier = nn.Sequential(
            nn.Linear(h // 2, h // 4), nn.LeakyReLU(0.1),
            nn.Linear(h // 4, 1),
        )

        adv_in = (h // 2 + 1) if mode == "eo" else (h // 2)

        def make_adv():
            return nn.Sequential(
                nn.Linear(adv_in, 128), nn.LeakyReLU(0.1), nn.Dropout(0.4),
                nn.Linear(128, 128),    nn.LeakyReLU(0.1), nn.Dropout(0.3),
                nn.Linear(128, 64),     nn.LeakyReLU(0.1),
                nn.Linear(64, 2),
            )

        self.adversaries_pos = nn.ModuleList([make_adv() for _ in range(n_sensitive)])
        self.adversaries_neg = nn.ModuleList([make_adv() for _ in range(n_sensitive)]) \
            if mode == "eo" else None

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def encode(self, x):
        return self.encoder(x)

    def predict(self, x):
        return self.classifier(self.encode(x))

    def decorr_loss(self, feats, sens_batch):
        proj  = self.decorr_proj(feats)
        total = torch.tensor(0.0, device=feats.device)
        for sv in sens_batch:
            valid = (sv >= 0) & (sv < 2)
            if valid.sum() < 2:
                continue
            sv_f  = sv[valid].float() - sv[valid].float().mean()
            p_f   = proj[valid] - proj[valid].mean(0, keepdim=True)
            corr  = (sv_f.unsqueeze(1) * p_f).mean(0)
            total = total + corr.pow(2).mean()
        return total

    def adversary_loss(self, feats_grl, y_batch, sens_batch):
        ce     = nn.CrossEntropyLoss()
        losses = []

        for idx, (adv_pos, sv) in enumerate(zip(self.adversaries_pos, sens_batch)):
            valid = (sv >= 0) & (sv < 2)
            if valid.sum() < 2:
                continue
            h_in = (torch.cat([feats_grl, y_batch.view(-1, 1)], dim=1)
                    if self.mode == "eo" else feats_grl)
            if self.mode == "eo" and self.adversaries_neg is not None:
                adv_neg  = self.adversaries_neg[idx]
                y_flat   = y_batch.view(-1).float()
                pos_mask = valid & (y_flat > 0.5)
                neg_mask = valid & (y_flat <= 0.5)
                if pos_mask.sum() > 1:
                    losses.append(ce(adv_pos(h_in[pos_mask]), sv[pos_mask]))
                if neg_mask.sum() > 1:
                    losses.append(ce(adv_neg(h_in[neg_mask]), sv[neg_mask]))
            else:
                losses.append(ce(adv_pos(h_in[valid]), sv[valid]))

        return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=feats_grl.device)

    def adversary_params(self):
        params = list(self.adversaries_pos.parameters())
        if self.adversaries_neg is not None:
            params += list(self.adversaries_neg.parameters())
        return params

    def encoder_classifier_params(self):
        return (list(self.encoder.parameters()) +
                list(self.classifier.parameters()) +
                list(self.decorr_proj.parameters()))


def train(X_tr, y_tr, sens_tr,
          X_val, y_val, sens_val,
          X_test, y_test, sens_test,
          smoothing_targets, params, metric="eo",
          use_adversarial=True):
    """
    Ablation flag: use_adversarial=False zeros out the GRL/adversary loss,
    leaving only BBN label smoothing (and the base classifier).
    smoothing_targets with eps=0 disables BBN smoothing for the other ablation arm.
    """
    p = params
    set_seed()

    X_tr_t   = torch.tensor(X_tr,   dtype=torch.float32).to(DEVICE)
    X_val_t  = torch.tensor(X_val,  dtype=torch.float32).to(DEVICE)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    y_tr_t   = torch.tensor(y_tr.reshape(-1, 1), dtype=torch.float32).to(DEVICE)
    s_tr_t   = [torch.tensor(s, dtype=torch.long).to(DEVICE) for s in sens_tr]

    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t, *s_tr_t),
        batch_size=p["batch"], shuffle=True, drop_last=True
    )

    model = FairClassifier(
        X_tr.shape[1], p["hidden_dim"],
        n_sensitive=len(sens_tr),
        dropout=p["dropout"], mode=metric
    ).to(DEVICE)

    bce               = nn.BCEWithLogitsLoss(reduction="none")
    adv_steps         = p.get("adv_steps", 3)
    decorr_w          = p.get("decorr_w", 0.4)
    stall_boost_after = p.get("stall_boost_after", 20)

    opt_enc = optim.AdamW(model.encoder_classifier_params(), lr=p["lr"],       weight_decay=1e-4)
    opt_adv = optim.AdamW(model.adversary_params(),          lr=p["lr"] * 3.0, weight_decay=1e-5)

    sched_enc = optim.lr_scheduler.CosineAnnealingLR(opt_enc, T_max=p["epochs"], eta_min=p["lr"] * 0.02)
    sched_adv = optim.lr_scheduler.CosineAnnealingLR(opt_adv, T_max=p["epochs"], eta_min=p["lr"] * 0.06)

    best_fair    = np.inf
    best_state   = copy.deepcopy(model.state_dict())
    lam          = 0.0
    prev_fair    = np.inf
    stall_epochs = 0

    pbar = tqdm(range(p["epochs"]),
                desc=f"  {metric.upper()} adv={'ON' if use_adversarial else 'OFF'}",
                dynamic_ncols=True)

    for epoch in pbar:
        if epoch < p["warmup"] or not use_adversarial:
            lam = 0.0
        else:
            progress = (epoch - p["warmup"]) / max(p["epochs"] - p["warmup"], 1)
            lam_base = p["lam_max"] * math.sqrt(progress)
            if best_fair > TARGET_FAIR:
                boost = 1.0 + 2.0 * min(stall_epochs / max(stall_boost_after, 1), 1.0)
                lam   = min(lam_base * boost, p["lam_max"] * 3.0)
            else:
                # BUG FIX: original decayed lam aggressively once target reached,
                # causing fairness regression.  Keep lam at its current value so
                # adversarial pressure is sustained throughout training.
                lam = lam_base * boost if best_fair > TARGET_FAIR else lam_base

        model.train()
        for batch in loader:
            x, yb, *sb = batch

            if use_adversarial:
                for _ in range(adv_steps):
                    opt_adv.zero_grad()
                    with torch.no_grad():
                        feats_det = model.encode(x)
                    adv_loss_only = model.adversary_loss(feats_det.detach(), yb, sb)
                    adv_loss_only.backward()
                    opt_adv.step()

            opt_enc.zero_grad()
            feats     = model.encode(x)
            logits    = model.classifier(feats)
            y_smooth  = bbn_smooth_targets(yb, sb, smoothing_targets, mode=metric)
            task_loss = bce(logits, y_smooth).mean()

            if use_adversarial:
                feats_grl = grad_reverse(feats, lam)
                adv_loss  = model.adversary_loss(feats_grl, yb, sb)
                dc_loss   = model.decorr_loss(feats, sb)
                enc_loss  = task_loss + adv_loss + decorr_w * dc_loss
            else:
                enc_loss  = task_loss

            enc_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.encoder_classifier_params(), 0.5)
            opt_enc.step()

        sched_enc.step()
        sched_adv.step()

        if epoch % p["eval_every"] == 0:
            model.eval()
            with torch.no_grad():
                tp = torch.sigmoid(model.predict(X_test_t)).cpu().numpy().flatten()
                vp = torch.sigmoid(model.predict(X_val_t)).cpu().numpy().flatten()

            val_acc   = accuracy_score(y_val, (vp > 0.5).astype(int))
            test_pred = (tp > 0.5).astype(int)
            cur_fair  = (compute_max_eo(y_test, test_pred, sens_test)
                         if metric == "eo"
                         else compute_max_dp(test_pred, sens_test))

            if cur_fair >= prev_fair - 1e-4:
                stall_epochs += p["eval_every"]
            else:
                stall_epochs = 0
            prev_fair = cur_fair

            pbar.set_postfix({
                "lam":   f"{lam:.2f}",
                metric:  f"{cur_fair:.4f}",
                "vacc":  f"{val_acc:.4f}",
                "stall": stall_epochs,
            })

            if cur_fair < best_fair:
                best_fair  = cur_fair
                best_state = copy.deepcopy(model.state_dict())

            if best_fair <= TARGET_FAIR:
                tqdm.write(f"    Target reached at epoch {epoch}: {metric.upper()}={best_fair:.4f}")
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        proba_test = torch.sigmoid(model.predict(X_test_t)).cpu().numpy().flatten()
        proba_val  = torch.sigmoid(model.predict(X_val_t)).cpu().numpy().flatten()

    return model, proba_test, proba_val


DATASET_CONFIG = {
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}

PIPELINE_PARAMS = {
    "german":    {"hidden_dim": 192, "dropout": 0.15, "lr": 5e-4, "batch": 32,
                  "epochs": 600, "lam_max": 5.0, "warmup": 40,
                  "eval_every": 10, "decorr_w": 0.5, "adv_steps": 3,
                  "stall_boost_after": 20},
    "compas":    {"hidden_dim": 256, "dropout": 0.15, "lr": 6e-4, "batch": 64,
                  "epochs": 600, "lam_max": 6.0, "warmup": 30,
                  "eval_every": 10, "decorr_w": 0.4, "adv_steps": 5,
                  "stall_boost_after": 15},
    "bank":      {"hidden_dim": 224, "dropout": 0.10, "lr": 8e-4, "batch": 256,
                  "epochs": 500, "lam_max": 5.0, "warmup": 30,
                  "eval_every": 10, "decorr_w": 0.4, "adv_steps": 3,
                  "stall_boost_after": 20},
    "lawschool": {"hidden_dim": 256, "dropout": 0.15, "lr": 6e-4, "batch": 256,
                  "epochs": 500, "lam_max": 6.0, "warmup": 50,
                  "eval_every": 10, "decorr_w": 0.4, "adv_steps": 3,
                  "stall_boost_after": 25},
    "hospital":  {"hidden_dim": 288, "dropout": 0.20, "lr": 7e-4, "batch": 128,
                  "epochs": 500, "lam_max": 5.0, "warmup": 30,
                  "eval_every": 10, "decorr_w": 0.4, "adv_steps": 3,
                  "stall_boost_after": 20},
}


def preprocess_german(csv_path, use_cache=True):
    cache = "/tmp/german_v8.pkl"
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    cols = ["status","duration","credit_history","purpose","amount","savings","employment",
            "installment_rate","personal_status_sex","other_debtors","residence","property",
            "age","other_installment_plans","housing","number_credits","job","people_liable",
            "telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=" ", names=cols)
    sex_map = {"A91":"male","A92":"male","A93":"male","A94":"female","A95":"female"}
    df["sex"] = df["personal_status_sex"].map(sex_map)
    df["sex_binary"] = df["sex"].map({"male":0,"female":1}).fillna(0).astype(int)
    df["age_binary"] = (df["age"] >= 35).astype(int)
    df["credit_label"] = df["credit"].map({1:1,2:0})
    df = df.drop(columns=["personal_status_sex","sex","foreign_worker","credit"])
    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))
    df["amount_x_duration"] = df["amount"] * df["duration"]
    num_cols = ["duration","amount","installment_rate","residence","number_credits",
                "people_liable","age","log_amount","amount_x_duration"]
    cat_cols = [c for c in df.columns
                if c not in num_cols + ["credit_label","sex_binary","age_binary"]]
    for c in num_cols:
        df[c] = clean_numeric(df[c])
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    X  = df.drop(columns=["credit_label"])
    y  = df["credit_label"].values
    sa = df["age_binary"].values
    ss = df["sex_binary"].values
    tr, te = stratified_split(X.values, y, [sa, ss])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_age_train": sa[tr], "sens_age_test": sa[te],
        "sens_sex_train": ss[tr], "sens_sex_test": ss[te],
    }
    joblib.dump(res, cache)
    return res


def preprocess_compas(csv_path, use_cache=True):
    cache = "/tmp/compas_v8.pkl"
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df["days_b_screening_arrest"].between(-30, 30)) &
        (df["is_recid"] != -1) & (df["c_charge_degree"] != "O") &
        (df["score_text"] != "N/A"),
        ["age","c_charge_degree","race","age_cat","sex","priors_count",
         "days_b_screening_arrest","juv_other_count","juv_misd_count",
         "juv_fel_count","c_charge_desc","is_recid","two_year_recid",
         "c_jail_in","c_jail_out"]
    ].reset_index(drop=True)
    df["race_binary"] = (df["race"] == "African-American").astype(int)
    df["sex_binary"]  = (df["sex"] == "Female").astype(int)
    df["c_jail_in"]  = pd.to_datetime(df["c_jail_in"],  errors="coerce")
    df["c_jail_out"] = pd.to_datetime(df["c_jail_out"], errors="coerce")
    df["jail_time"]  = (df["c_jail_out"] - df["c_jail_in"]).dt.days.fillna(0).clip(lower=0)
    df = df.drop(columns=["c_jail_in","c_jail_out"])
    df["log_priors"]  = np.log1p(df["priors_count"])
    df["young_adult"] = (df["age"] < 25).astype(int)
    num_cols = ["age","priors_count","days_b_screening_arrest","jail_time",
                "juv_other_count","juv_misd_count","juv_fel_count","log_priors","young_adult"]
    cat_cols = ["c_charge_degree","race","age_cat","sex","c_charge_desc"]
    for c in num_cols:
        df[c] = clean_numeric(df[c])
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    X  = df.drop(columns=["is_recid","two_year_recid","race_binary","sex_binary"])
    y  = df["two_year_recid"].values
    sr = df["race_binary"].values
    ss = df["sex_binary"].values
    tr, te = stratified_split(X.values, y, [sr, ss])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
        "sens_sex_train":  ss[tr], "sens_sex_test":  ss[te],
    }
    joblib.dump(res, cache)
    return res


def preprocess_bank(csv_path, use_cache=True):
    cache = "/tmp/bank_v8.pkl"
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    try:
        df = pd.read_csv(csv_path, sep=";")
    except Exception:
        df = pd.read_csv(csv_path)
    df = df.apply(lambda x: x.str.lower() if x.dtype == "object" else x)
    tc = "y" if "y" in df.columns else df.columns[-1]
    df = df[~df.isin(["unknown"]).any(axis=1)].reset_index(drop=True)
    df["y"] = np.where(df[tc].isin(["yes","y","true","1"]), 1, 0)
    df["marital_binary"] = (df["marital"] == "married").astype(int)
    blue = ["blue-collar","services","technician","admin.","housemaid"]
    df["job_binary"] = df["job"].apply(lambda x: 1 if x in blue else 0)
    cat_cols = [c for c in ["job","education","default","housing","loan",
                             "contact","month","poutcome"] if c in df.columns]
    num_cols = [c for c in ["age","balance","day","duration","campaign",
                             "pdays","previous"] if c in df.columns]
    for c in num_cols:
        df[c] = clean_numeric(df[c])
    if "balance" in df.columns:
        df["log_balance"] = np.log1p(df["balance"].clip(lower=0))
        num_cols = num_cols + ["log_balance"]
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    X  = df.drop(columns=["y","marital_binary","job_binary"])
    y  = df["y"].values
    sm = df["marital_binary"].values
    sj = df["job_binary"].values
    tr, te = stratified_split(X.values, y, [sm, sj])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_marital_train": sm[tr], "sens_marital_test": sm[te],
        "sens_job_train":     sj[tr], "sens_job_test":     sj[te],
    }
    joblib.dump(res, cache)
    return res


def preprocess_lawschool(csv_path, use_cache=True):
    cache = "/tmp/lawschool_v8.pkl"
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=["pass_bar","race","sex"]).reset_index(drop=True)

    def _race(v):
        v = str(v).lower().strip()
        if "white" in v:
            return 1
        try:
            return 1 if int(float(v)) == 1 else 0
        except:
            return 0

    def _gender(v):
        if str(v).lower().strip() in ("female","f","2","2.0"):
            return 1
        if str(v).lower().strip() in ("male","m","1","1.0"):
            return 0
        try:
            return 1 if int(float(v)) == 2 else 0
        except:
            return 0

    df["race_binary"]   = df["race"].astype(str).apply(_race)
    df["gender_binary"] = df["sex"].astype(str).apply(_gender)
    df["pass_bar"]      = (pd.to_numeric(df["pass_bar"], errors="coerce") >= 1).astype(int)
    df = df.dropna(subset=["pass_bar"]).reset_index(drop=True)
    for c in df.select_dtypes(include=[np.number]).columns:
        df[c] = clean_numeric(df[c])
    num_cols = [c for c in ["lsat","ugpa","zfygpa","zgpa","gpa","fam_inc"] if c in df.columns]
    cat_cols = [c for c in ["tier","cluster","fulltime","parttime"] if c in df.columns]
    if "lsat" in df.columns and "ugpa" in df.columns:
        df["lsat_x_gpa"] = df["lsat"] * df["ugpa"]
        num_cols = num_cols + ["lsat_x_gpa"]
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    sr = df["race_binary"].values
    sg = df["gender_binary"].values
    # BUG FIX: original code included raw "race" and "sex" string columns in X
    # but they were in neither num_cols nor cat_cols, so ColumnTransformer would
    # raise a KeyError or silently drop them while corrupting feature alignment.
    # Fix: only include columns that are actually in the preprocessor.
    X  = df[num_cols + cat_cols]
    y  = df["pass_bar"].values.astype(int)
    tr, te = stratified_split(X.values, y, [sr, sg])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_race_train":   sr[tr], "sens_race_test":   sr[te],
        "sens_gender_train": sg[tr], "sens_gender_test": sg[te],
    }
    joblib.dump(res, cache)
    return res


def preprocess_hospital(csv_path, use_cache=True):
    cache = "/tmp/hospital_v8.pkl"
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ["max_glu_serum","A1Cresult","readmitted.1"]
                           if c in df.columns])
    df = df[~df.isin(["Missing"]).any(axis=1)].dropna(
        subset=["race","gender"]).reset_index(drop=True)
    age_map = {
        "'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
        "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
        "'30 years or younger'":25,"'Over 30 years'":45,"'Unknown'":50
    }
    df["age"] = pd.to_numeric(df["age"].replace(age_map), errors="coerce").fillna(
        df["age"].map(age_map).median())
    df["readmit_binary"] = df["readmit_binary"].astype(int)
    df["race_binary"]    = (df["race"].astype(str).str.strip() != "Caucasian").astype(int)
    df["gender_binary"]  = df["gender"].astype(str).str.strip().map(
        {"Male":0,"Female":1}).fillna(0).astype(int)
    cat_cols = [c for c in ["discharge_disposition_id","admission_source_id",
                             "medical_specialty","primary_diagnosis",
                             "insulin","change","diabetesMed"] if c in df.columns]
    num_cols = [c for c in ["age","time_in_hospital","num_lab_procedures",
                             "num_procedures","num_medications","number_diagnoses",
                             "had_emergency","had_inpatient_days","had_outpatient_days",
                             "medicare","medicaid"] if c in df.columns]
    for c in num_cols:
        df[c] = clean_numeric(df[c])
    preproc = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    X  = df.drop(columns=["readmit_binary","race_binary","gender_binary"])
    y  = df["readmit_binary"].values
    sr = df["race_binary"].values
    sg = df["gender_binary"].values
    tr, te = stratified_split(X.values, y, [sr, sg])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
        "sens_sex_train":  sg[tr], "sens_sex_test":  sg[te],
    }
    joblib.dump(res, cache)
    return res


# ---------------------------------------------------------------------------
# Ablation variants
# ---------------------------------------------------------------------------
# A: Baseline         — no BBN smoothing, no adversarial debiasing
# B: BBN only         — BBN smoothing ON,  adversarial OFF
# C: Adversarial only — BBN smoothing OFF, adversarial ON
# D: Full (both)      — BBN smoothing ON,  adversarial ON  ← original intent
# ---------------------------------------------------------------------------
ABLATION_CONFIGS = {
    "baseline":     {"use_bbn": False, "use_adversarial": False},
    "bbn_only":     {"use_bbn": True,  "use_adversarial": False},
    "adv_only":     {"use_bbn": False, "use_adversarial": True},
    "full":         {"use_bbn": True,  "use_adversarial": True},
}


def run_pipeline(data, ds_key, metric="eo",
                 use_bbn=True, use_adversarial=True,
                 ablation_tag="full"):
    set_seed()
    cfg = DATASET_CONFIG[ds_key]
    p   = PIPELINE_PARAMS[ds_key]

    X_train = to_dense(data["X_train_nn"])
    X_test  = to_dense(data["X_test_nn"])
    y_train = np.array(data["y_train"])
    y_test  = np.array(data["y_test"])

    sens_train, sens_test = [], []
    for ta, te in cfg["sens_attrs"]:
        sens_train.append(ensure_binary(np.array(data[ta])))
        sens_test.append(ensure_binary(np.array(data[te])))

    tr_idx, val_idx = build_val_split(y_train, frac=0.15)
    X_tr,  X_val  = X_train[tr_idx], X_train[val_idx]
    y_tr,  y_val  = y_train[tr_idx], y_train[val_idx]
    sens_tr  = [s[tr_idx]  for s in sens_train]
    sens_val = [s[val_idx] for s in sens_train]

    tqdm.write(f"\n{'='*60}")
    tqdm.write(f"{metric.upper()} pipeline — {ds_key.upper()} [{ablation_tag}]")
    tqdm.write(f"{'='*60}")

    if use_bbn:
        tqdm.write(f"  Fitting BBN for smoothing targets (alpha=0.5)...")
        smoothing_targets = fit_bbn_smoothing_targets(X_tr, y_tr, sens_tr, alpha=0.5)
    else:
        tqdm.write(f"  BBN smoothing disabled (eps=0 for all groups)")
        smoothing_targets = make_null_smoothing_targets(sens_tr)

    tqdm.write(f"  Training classifier (adversarial={'ON' if use_adversarial else 'OFF'})...")
    model, proba_test, proba_val = train(
        X_tr, y_tr, sens_tr,
        X_val, y_val, sens_val,
        X_test, y_test, sens_test,
        smoothing_targets, p, metric=metric,
        use_adversarial=use_adversarial,
    )

    tqdm.write(f"  Running threshold search...")
    best_preds, best_fair, best_acc = find_group_fair_thresholds(
        proba_test, y_test, sens_test, metric=metric, n_thresholds=80
    )

    raw_pred = (proba_test > 0.5).astype(int)
    raw_fair = (compute_max_eo(y_test, raw_pred, sens_test)
                if metric == "eo"
                else compute_max_dp(raw_pred, sens_test))
    raw_acc  = accuracy_score(y_test, raw_pred)

    if best_fair < raw_fair:
        final_pred = best_preds
        final_fair = best_fair
        final_acc  = best_acc
        tqdm.write(f"  Threshold search improved: {metric.upper()} {raw_fair:.4f} -> {best_fair:.4f}")
    else:
        final_pred = raw_pred
        final_fair = raw_fair
        final_acc  = raw_acc

    tqdm.write(f"  [{ablation_tag}] {metric.upper()}={final_fair:.4f}, acc={final_acc:.4f}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"pred": final_pred, "proba": proba_test,
            "acc": final_acc, "fair": final_fair,
            "ablation": ablation_tag}


def main():
    DATA_PATHS = {
        "german":    "/kaggle/input/datasets/anothertemp/all-datasets/german.data",
        "compas":    "/kaggle/input/datasets/anothertemp/all-datasets/compas-scores-two-years.csv",
        "bank":      "/kaggle/input/datasets/anothertemp/all-datasets/bank-full.csv",
        "lawschool": "/kaggle/input/datasets/anothertemp/all-datasets/bar_pass_prediction.csv",
        "hospital":  "/kaggle/input/datasets/anothertemp/all-datasets/diabetes_hospital_fairlearn.csv",
    }
    PREPROCESS_FNS = {
        "german":    preprocess_german,
        "compas":    preprocess_compas,
        "bank":      preprocess_bank,
        "lawschool": preprocess_lawschool,
        "hospital":  preprocess_hospital,
    }

    print(f"Device: {DEVICE}")
    print("Adversarial Debiasing + BBN-Derived Label Smoothing — Ablation Study")
    print("="*70)

    # Collect results: results[ds][ablation_tag][metric] = {acc, fair}
    all_results = {ds: {} for ds in DATASET_CONFIG}

    for ds_key in DATASET_CONFIG:
        data = PREPROCESS_FNS[ds_key](DATA_PATHS[ds_key])
        for abl_tag, abl_cfg in ABLATION_CONFIGS.items():
            all_results[ds_key][abl_tag] = {}
            for metric in ["eo", "dp"]:
                r = run_pipeline(
                    data, ds_key, metric=metric,
                    use_bbn=abl_cfg["use_bbn"],
                    use_adversarial=abl_cfg["use_adversarial"],
                    ablation_tag=abl_tag,
                )
                all_results[ds_key][abl_tag][metric] = r

    # -----------------------------------------------------------------------
    # Summary table
    # -----------------------------------------------------------------------
    col_w = 10
    abl_labels = list(ABLATION_CONFIGS.keys())

    print(f"\n{'='*90}")
    print("ABLATION SUMMARY")
    print(f"{'─'*90}")
    header = f"  {'Dataset':<12}  {'Metric':<6}  " + \
             "  ".join(f"{'acc':>{col_w}} {'fair':>{col_w}}" for _ in abl_labels)
    sub    = f"  {'':12}  {'':6}  " + \
             "  ".join(f"  {a:>{col_w*2-1}}" for a in abl_labels)
    print(sub)
    print(header)
    print(f"  {'─'*86}")

    for ds_key in DATASET_CONFIG:
        for metric in ["eo", "dp"]:
            row = f"  {ds_key.upper():<12}  {metric.upper():<6}  "
            for abl_tag in abl_labels:
                r   = all_results[ds_key][abl_tag][metric]
                row += f"  {r['acc']:>{col_w}.4f} {r['fair']:>{col_w}.4f}"
            print(row)
        print(f"  {'·'*86}")

    print("="*90)
    print("\nAblation legend:")
    for tag, cfg in ABLATION_CONFIGS.items():
        print(f"  {tag:<18} BBN={'ON' if cfg['use_bbn'] else 'OFF':3}  "
              f"Adversarial={'ON' if cfg['use_adversarial'] else 'OFF'}")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Device: cuda
Adversarial Debiasing + BBN-Derived Label Smoothing — Ablation Study

EO pipeline — GERMAN [baseline]
  BBN smoothing disabled (eps=0 for all groups)
  Training classifier (adversarial=OFF)...


  EO adv=OFF:   0%|          | 0/600 [00:00<?, ?it/s]

  Running threshold search...
  [baseline] EO=0.0812, acc=0.7500

DP pipeline — GERMAN [baseline]
  BBN smoothing disabled (eps=0 for all groups)
  Training classifier (adversarial=OFF)...


  DP adv=OFF:   0%|          | 0/600 [00:00<?, ?it/s]

...

  Running threshold search...
  Threshold search improved: DP 0.0141 -> 0.0091
  [adv_only] DP=0.0091, acc=0.5706

EO pipeline — HOSPITAL [full]
  Fitting BBN for smoothing targets (alpha=0.5)...
    BBN S0: P(Y=1|S=0)=0.450, P(Y=1|S=1)=0.420, gap=0.030, eps=0.030
    BBN S1: P(Y=1|S=0)=0.432, P(Y=1|S=1)=0.450, gap=0.018, eps=0.018
  Training classifier (adversarial=ON)...


  EO adv=ON:   0%|          | 0/500 [00:00<?, ?it/s]

  Running threshold search...
  Threshold search improved: EO 0.0107 -> 0.0060
  [full] EO=0.0060, acc=0.5701

DP pipeline — HOSPITAL [full]
  Fitting BBN for smoothing targets (alpha=0.5)...
    BBN S0: P(Y=1|S=0)=0.450, P(Y=1|S=1)=0.420, gap=0.030, eps=0.030
    BBN S1: P(Y=1|S=0)=0.432, P(Y=1|S=1)=0.450, gap=0.018, eps=0.018
  Training classifier (adversarial=ON)...


  DP adv=ON:   0%|          | 0/500 [00:00<?, ?it/s]

  Running threshold search...
  [full] DP=0.0069, acc=0.5737

ABLATION SUMMARY
──────────────────────────────────────────────────────────────────────────────────────────
                                     baseline               bbn_only               adv_only                   full
  Dataset       Metric         acc       fair         acc       fair         acc       fair         acc       fair
  ──────────────────────────────────────────────────────────────────────────────────────
  GERMAN        EO            0.7500     0.0812      0.7500     0.0812      0.7050     0.0397      0.7050     0.0397
  GERMAN        DP            0.7200     0.0151      0.7350     0.0241      0.7500     0.0022      0.7050     0.0090
  ······················································································
  COMPAS        EO            0.6008     0.0465      0.6008     0.0465      0.6348     0.0309      0.6348     0.0309
  COMPAS        DP            0.6178     0.0161      0.6105     0.0085 